# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/payalpawar4/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
# Install DuckDB for querying the large warehouse
%pip -q install duckdb

import duckdb
from google.colab import userdata

# Get the Hugging Face token safely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Give DuckDB access to the gated Hugging Face dataset
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# We will develop on the March 2026 partition
TABLE_PATH = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
)

print("Connection successful.")
print("Using March 2026 warehouse partition.")

Connection successful.
Using March 2026 warehouse partition.


In [8]:
TABLE_PATH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print(TABLE_PATH)


read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)



In [11]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

HF_TOKEN loaded: True
Token length: 37


In [12]:
from huggingface_hub import whoami

info = whoami(token=HF_TOKEN)

print("Hugging Face user:", info["name"])

Hugging Face user: Payal0401


In [13]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected to Hugging Face.")


DuckDB connected to Hugging Face.


In [14]:
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {TABLE_PATH}
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
# Query 1 — Verify the grain:
# one row should represent one report_date + client + content item.

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS distinct_grain_rows,
    COUNT(*) - COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS duplicate_grain_rows
FROM {TABLE_PATH}
""").df()

display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_grain_rows
0,9841378,9841378,0


In [16]:
# Query 2 — March row count and observed date span

date_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM {TABLE_PATH}
WHERE month = '2026-03'
""").df()

display(date_check)

,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


In [17]:
# Query 3 — Availability check using IS TRUE

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        AS ga4_available_rows
FROM {TABLE_PATH}
WHERE month = '2026-03'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [18]:
# Five-feature frame for March 2026
# These are features available in the warehouse observation.

features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    sessions_organic,
    scroll_events
FROM {TABLE_PATH}
WHERE month = '2026-03'
LIMIT 20
""").df()

display(features)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,sessions_organic,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,<NA>,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,7.347280,<NA>,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,7.832461,<NA>,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,3.272727,<NA>,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,5.636364,<NA>,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,4.500000,<NA>,<NA>,<NA>


In [19]:
# Deliberate leakage experiment
# gsc_clicks is the label, so using it as a feature is intentionally wrong.

leaky_frame = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    sessions_organic,
    scroll_events,
    gsc_clicks
FROM {TABLE_PATH}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_clicks IS NOT NULL
LIMIT 1000
""").df()

display(leaky_frame.head())

,gsc_impressions,gsc_avg_position,ga4_sessions,sessions_organic,scroll_events,gsc_clicks
0,20,3.350000,<NA>,<NA>,<NA>,0
1,1,0.000000,<NA>,<NA>,<NA>,0
2,125,4.928000,<NA>,<NA>,<NA>,1
3,7,4.000000,<NA>,<NA>,<NA>,0
4,11,2.272727,<NA>,<NA>,<NA>,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.